# Chunk Visualization for the Sentinel-2 HDF5 Tile T29TPG

Inspects the spatial layout of HDF5 chunks for tile T29TPG and characterises per timestamp data completeness, following the same structure as the T29TQG reference notebook.

Contents:
1. HDF5 metadata dump (shape, chunks, dtype, attributes).
2. Per chunk bounding boxes derived from `xs_new`/`ys_new`.
3. Tile dimensions and per chunk bounds table.
4. Map of all chunks on an OpenStreetMap basemap.
5. Anatomy of a single chunk.
6. Reference 256 by 256 pixel grid overlay.
7. Chunks per timestamp.
8. Per timestamp data completeness and usable scene calendar.
9. Per timestamp valid vs invalid pixel panels for 20 consecutive timestamps (5 cols by 4 rows).
10. Vegetation indices time series for a sample chunk.

In [ ]:
# Install deps (idempotent)
!pip install -q h5py geopandas shapely contextily fiona matplotlib numpy

In [ ]:
# Imports
import os, sys, platform
for _v in ("MallocStackLogging", "MallocStackLoggingNoCompact", "MALLOC_STACK_LOGGING"):
    os.environ.pop(_v, None)

import numpy as np
import pandas as pd
import h5py
import geopandas as gpd
from shapely.geometry import box
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Patch
from matplotlib.cm import get_cmap
from matplotlib.colors import ListedColormap, BoundaryNorm
import contextily as ctx
from datetime import datetime, timedelta

print(f"Python {platform.python_version()}  |  h5py {h5py.__version__}  |  numpy {np.__version__}")

In [ ]:
# Paths and constants
HDF5_PATH = "/Users/dgonzales22/Documents/Research/veg-s2s/T29TPG/T29TPG.h5"
OUT_DIR   = "/Users/dgonzales22/Documents/Research/veg-s2s/T29TPG/figures"
os.makedirs(OUT_DIR, exist_ok=True)

PIXEL_SIZE_M       = 10.0
TILE_CRS           = "EPSG:32629"   # UTM zone 29N, matches T29TPG
BUFFER_M           = 5_000
NODATA_VAL         = 65535
XYS_SENTINEL       = -9999          # xs_new/ys_new value for unmasked positions
BAND_FOR_VALIDITY  = 0

print(f"HDF5 : {HDF5_PATH}")

---
## 1. HDF5 metadata dump

In [ ]:
def inspect_hdf5(path):
    with h5py.File(path, "r") as f:
        print(f"File: {path}\n")
        print("Root attributes:")
        for k, v in f.attrs.items():
            s = str(v).replace("\n", " ")
            print(f"  {k:20s} = {s[:100] + ('...' if len(s) > 100 else '')}")
        print("\nDatasets:")
        def show(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  {name:30s} shape={str(obj.shape):20s} dtype={obj.dtype}  "
                      f"chunks={obj.chunks}  compression={obj.compression}")
        f.visititems(show)

inspect_hdf5(HDF5_PATH)

In [ ]:
# Pull the constants we need throughout the notebook
with h5py.File(HDF5_PATH, "r") as f:
    A = dict(f.attrs)
    band_names   = [b.decode() if isinstance(b, bytes) else b for b in A["band_names"]]
    chip_size    = int(A["chip_size"])           # 256
    pixel_res    = float(A["pixel_res"])         # 10 m
    TILE_MIN_X   = float(A["bounds_left"])
    TILE_MAX_X   = float(A["bounds_right"])
    TILE_MIN_Y   = float(A["bounds_bottom"])
    TILE_MAX_Y   = float(A["bounds_top"])

    values_shape  = f["values"].shape            # (T, B, N)
    values_chunks = f["values"].chunks           # (chunk_t, B, chunk_pixels)
    n_timestamps, n_bands, n_pixels = values_shape

    timestamps = pd.to_datetime(f["original_timestamps"][:], unit="ms")
    cloud_cover_pct      = f["cloud_cover_pt"][:]
    clear_pixel_count    = f["clear_pixel_count_pt"][:]
    pixel_count_pt       = f["pixel_count_pt"][:]
    orbit_pixel_count    = f["count_orbit_pixels_pt"][:]
    chip_x_bin           = f["chip_x_bin"][:]
    chip_y_bin           = f["chip_y_bin"][:]
    chip_pixel_count_arr = f["chip_pixel_count"][:]

chunk_pixel_len = values_chunks[2]
chip_m  = chip_size * pixel_res                   # 2560 m on the ground
n_chunks = int(np.ceil(n_pixels / chunk_pixel_len))

print(f"Bands ({n_bands}): {band_names}")
print(f"Timestamps: {n_timestamps}   {timestamps.min().date()} to {timestamps.max().date()}")
print(f"Pixel axis: {n_pixels:,}  |  chunk size on pixel axis: {chunk_pixel_len:,}  |  chunks: {n_chunks}")
print(f"Chip size : {chip_size} px = {chip_m:.0f} m on the ground")

---
## 2. Per chunk spatial bboxes

In this file the pixel axis chunk size equals one 256 by 256 chip (65,536 pixels). Each chunk therefore corresponds to exactly one chip, and the file holds 959 chips. `xs_new` and `ys_new` are UTM 29N coordinates in metres, with a `-9999` sentinel for unmasked positions which must be excluded when reading per chunk extents.

In [ ]:
def compute_chunk_bboxes(hdf5_path, chunk_size=chunk_pixel_len):
    with h5py.File(hdf5_path, "r") as f:
        xs = f["xs_new"][:]
        ys = f["ys_new"][:]
    n_chunks_local = int(np.ceil(len(xs) / chunk_size))
    bboxes = []
    for cid in range(n_chunks_local):
        lo = cid * chunk_size
        hi = min(lo + chunk_size, len(xs))
        sx = xs[lo:hi]; sy = ys[lo:hi]
        m  = (sx != XYS_SENTINEL) & (sy != XYS_SENTINEL)
        if not m.any():
            continue
        bboxes.append({
            "chunk_id": cid,
            "minx": float(sx[m].min()),
            "maxx": float(sx[m].max()),
            "miny": float(sy[m].min()),
            "maxy": float(sy[m].max()),
            "n_pixels": int(m.sum()),
        })
    return bboxes, xs, ys

print("Computing chunk bboxes ...")
CHUNK_BBOXES, xs_global, ys_global = compute_chunk_bboxes(HDF5_PATH)
print(f"  -> {len(CHUNK_BBOXES)} chunks with valid pixels")

---
## 3. Tile dimensions and per chunk bounds

In [ ]:
tile_w = TILE_MAX_X - TILE_MIN_X
tile_h = TILE_MAX_Y - TILE_MIN_Y

print("=" * 70)
print("  TILE DIMENSIONS")
print("=" * 70)
print(f"  Easting  (X) : [{TILE_MIN_X:>12,.0f} ... {TILE_MAX_X:>12,.0f}]   width  = {tile_w:>10,.0f} m  ({tile_w/1000:6.1f} km)")
print(f"  Northing (Y) : [{TILE_MIN_Y:>12,.0f} ... {TILE_MAX_Y:>12,.0f}]   height = {tile_h:>10,.0f} m  ({tile_h/1000:6.1f} km)")
print(f"  Total area   : {tile_w * tile_h / 1e6:>10,.1f} km^2")
print(f"  CRS          : {TILE_CRS}")

df_chunks = pd.DataFrame(CHUNK_BBOXES)
df_chunks["width_km"]  = (df_chunks.maxx - df_chunks.minx) / 1000
df_chunks["height_km"] = (df_chunks.maxy - df_chunks.miny) / 1000
df_chunks.head(10)

---
## 4. Map of all chunks on OpenStreetMap

In [ ]:
def plot_tile_with_chunks(chunk_bboxes, save_name="T29TPG_chunks_map.png"):
    plot_min_x = TILE_MIN_X - BUFFER_M
    plot_max_x = TILE_MAX_X + BUFFER_M
    plot_min_y = TILE_MIN_Y - BUFFER_M
    plot_max_y = TILE_MAX_Y + BUFFER_M

    fig, ax = plt.subplots(figsize=(11, 11 * tile_h / max(tile_w, 1)))
    palette = get_cmap("tab20", 20)

    for cb in chunk_bboxes:
        color = palette(cb["chunk_id"] % palette.N)
        ax.add_patch(Rectangle(
            (cb["minx"], cb["miny"]),
            cb["maxx"] - cb["minx"],
            cb["maxy"] - cb["miny"],
            facecolor=color, edgecolor="black", lw=0.3, alpha=0.55,
        ))

    ax.add_patch(Rectangle(
        (TILE_MIN_X, TILE_MIN_Y), tile_w, tile_h,
        facecolor="none", edgecolor="black", lw=2.0,
    ))

    ax.set_xlim(plot_min_x, plot_max_x)
    ax.set_ylim(plot_min_y, plot_max_y)
    ax.set_aspect("equal")
    try:
        ctx.add_basemap(ax, crs=TILE_CRS, source=ctx.providers.OpenStreetMap.Mapnik, attribution_size=6)
    except Exception as exc:
        print(f"OSM basemap unavailable ({exc}); rendering without basemap")
    ax.set_xlabel("Easting (m, UTM 29N)")
    ax.set_ylabel("Northing (m, UTM 29N)")
    ax.set_title(f"T29TPG chunks on OSM basemap ({len(chunk_bboxes)} chunks)")
    plt.tight_layout()
    out = os.path.join(OUT_DIR, save_name)
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved {out}")
    plt.show()

plot_tile_with_chunks(CHUNK_BBOXES)

---
## 5. Anatomy of a single chunk

Each chunk holds one chip across 10 spectral bands. The diagram shows the band stack and total chunk size in bytes.

In [ ]:
band_colors = ["#a3cb38", "#c97a7a", "#8b3a3a", "#7f8c8d",
               "#4c72b0", "#7e57a0", "#3a8a7a", "#c8923a",
               "#34495e", "#2c3e50"]
RED_IDX = band_names.index("B4")
NIR_IDX = band_names.index("B8")

fig, ax = plt.subplots(figsize=(10, 6.5))
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.10); ax.axis("off")
band_h = 0.85 / n_bands
for i, (bn, bc) in enumerate(zip(band_names, band_colors[:n_bands])):
    y = 0.90 - (i + 1) * band_h
    ax.add_patch(Rectangle((0.05, y), 0.9, band_h * 0.85,
                           facecolor=bc, edgecolor="black", lw=0.5, alpha=0.80))
    if i in (RED_IDX, NIR_IDX):
        ax.add_patch(Rectangle((0.05, y), 0.9, band_h * 0.85,
                               facecolor="none", edgecolor="#c8923a", lw=1.8))
    ax.text(0.02, y + band_h * 0.42, bn, ha="right", va="center",
            fontsize=10, fontweight="bold")
    ax.text(0.5,  y + band_h * 0.42,
            f"{chunk_pixel_len:,} pixels (uint16, 2 B each)",
            ha="center", va="center", fontsize=9, color="white", fontweight="bold")

chunk_bytes = n_bands * chunk_pixel_len * 2
ax.text(0.5, 0.02,
        f"One chunk = 1 timestep slice x {n_bands} bands x {chunk_pixel_len:,} pixels = {chunk_bytes/1e6:.2f} MB",
        ha="center", fontsize=11, fontweight="bold")
ax.set_title("Anatomy of a single HDF5 chunk in T29TPG.h5")
plt.tight_layout(); plt.show()

---
## 6. 256 by 256 pixel grid overlay

What a 256 by 256 patch grid looks like on the tile extent at 10 m resolution. Each block covers 2.56 km by 2.56 km. In this file the pixel chunks already match this geometry, so the grid below also represents one chunk per cell.

In [ ]:
patch_m = chip_size * PIXEL_SIZE_M
n_cols = int(np.ceil((TILE_MAX_X - TILE_MIN_X) / patch_m))
n_rows = int(np.ceil((TILE_MAX_Y - TILE_MIN_Y) / patch_m))
bytes_per_block = n_bands * chip_size * chip_size * 2

print(f"Grid       : {n_rows} rows x {n_cols} cols = {n_rows * n_cols:,} blocks")
print(f"Block size : {chip_size} x {chip_size} px  ->  {patch_m/1000:.2f} km x {patch_m/1000:.2f} km")
print(f"Bytes/block: {bytes_per_block/1e6:.2f} MB (uint16 x {n_bands} bands x {chip_size*chip_size:,} px)")

fig, ax = plt.subplots(figsize=(11, 11 * tile_h / max(tile_w, 1)))
for r in range(n_rows):
    for c in range(n_cols):
        x = TILE_MIN_X + c * patch_m
        y = TILE_MAX_Y - (r + 1) * patch_m
        w = min(patch_m, TILE_MAX_X - x)
        h = min(patch_m, TILE_MAX_Y - y)
        ax.add_patch(Rectangle((x, y), w, h, facecolor="none", edgecolor="#2c3e50", lw=0.3))
ax.add_patch(Rectangle((TILE_MIN_X, TILE_MIN_Y), tile_w, tile_h,
                       facecolor="none", edgecolor="black", lw=2.0))
ax.set_xlim(TILE_MIN_X - BUFFER_M, TILE_MAX_X + BUFFER_M)
ax.set_ylim(TILE_MIN_Y - BUFFER_M, TILE_MAX_Y + BUFFER_M)
ax.set_aspect("equal")
try:
    ctx.add_basemap(ax, crs=TILE_CRS, source=ctx.providers.OpenStreetMap.Mapnik, attribution_size=6)
except Exception as exc:
    print(f"OSM basemap unavailable ({exc})")
ax.set_title(f"256x256 patch grid on T29TPG ({n_rows*n_cols:,} blocks)")
ax.set_xlabel("Easting (m, UTM 29N)"); ax.set_ylabel("Northing (m, UTM 29N)")
plt.tight_layout(); plt.show()

---
## 7. Chunks per timestamp

The `values` dataset is chunked as `(chunk_t, n_bands, chunk_pixel_len) = (12, 10, 65536)`. The pixel axis layout is shared across timestamps, so every timestamp has the same set of chunk extents.

In [ ]:
print(f"values shape   : {values_shape}  (T={n_timestamps}, B={n_bands}, N={n_pixels:,})")
print(f"values chunks  : {values_chunks}")
print(f"chunks per ts  : {n_chunks}  (one per 256x256 chip)")
print(f"time chunking  : 1 chunk along time axis covers {values_chunks[0]} timestamps")

---
## 8. Per timestamp data completeness and usable scene calendar

Per scene pixel accounting using the `_pt` arrays. `pixel_count_pt` is a constant equal to the in AOI pixel count, so `clear / total` measures the cloud free fraction of the tile and `orbit / total` measures the swath coverage. Scenes are labelled by combining the two ratios, then plotted as a year long calendar ribbon.

In [ ]:
df_complete = pd.DataFrame({
    "date": timestamps,
    "cloud_cover_pct": cloud_cover_pct,
    "total_px": pixel_count_pt,
    "orbit_px": orbit_pixel_count,
    "clear_px": clear_pixel_count,
}).sort_values("date").reset_index(drop=True)
df_complete["swath_coverage"]   = df_complete["orbit_px"] / df_complete["total_px"]
df_complete["valid_over_total"] = df_complete["clear_px"] / df_complete["total_px"]
df_complete["valid_over_swath"] = df_complete["clear_px"] / df_complete["orbit_px"].replace(0, np.nan)

def classify(r):
    if r.swath_coverage >= 0.95 and r.cloud_cover_pct <= 10: return "full_clear"
    if r.swath_coverage < 0.50: return "edge_pass"
    if r.swath_coverage >= 0.50 and r.cloud_cover_pct <= 20: return "partial_clear"
    return "clouded"
df_complete["category"] = df_complete.apply(classify, axis=1)

print(df_complete["category"].value_counts().reindex(["full_clear","partial_clear","edge_pass","clouded"], fill_value=0))
df_complete.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_complete["date"], df_complete["swath_coverage"]*100,   "o-", label="Swath coverage (orbit/total)", color="steelblue")
ax.plot(df_complete["date"], df_complete["valid_over_total"]*100, "o-", label="Valid / Total (clear/total)",  color="seagreen")
ax.plot(df_complete["date"], df_complete["valid_over_swath"]*100, "o--",label="Valid / Swath (clear/orbit)", color="darkorange", alpha=0.7)
ax.axhline(80, color="gray", ls=":", lw=1, label="80% threshold")
ax.set_ylabel("Completeness (%)"); ax.set_xlabel("Date"); ax.set_ylim(0, 105)
ax.set_title("Data completeness per Sentinel-2 acquisition, T29TPG 2025")
ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
palette = {"full_clear":"#2ca02c","partial_clear":"#bcbd22","edge_pass":"#1f77b4","clouded":"#7f7f7f"}
fig, ax = plt.subplots(figsize=(12, 1.8))
for _, r in df_complete.iterrows():
    ax.axvspan(r.date, r.date + pd.Timedelta(days=1), color=palette[r.category], alpha=0.9)
ax.set_xlim(df_complete.date.min() - pd.Timedelta(days=2), df_complete.date.max() + pd.Timedelta(days=2))
ax.set_yticks([])
ax.set_title("Usable scene calendar (T29TPG, 2025)")
handles = [plt.Rectangle((0,0),1,1,color=c) for c in palette.values()]
ax.legend(handles, list(palette.keys()), loc="upper center", bbox_to_anchor=(0.5,-0.2), ncol=4, frameon=False)
plt.tight_layout(); plt.show()

---
## 9. Valid vs invalid pixel panels (20 consecutive timestamps)

Renders a 5 by 4 grid of consecutive acquisitions. Each panel shows a coarse valid vs invalid heatmap with the persistent AOI footprint overlaid as a black outline. Two sources of invalid pixels:

1. Outside the stored mask, positions not present in `xs_new`/`ys_new`, the file's permanent footprint.
2. Nodata in `values`, positions present in the mask whose band 0 value equals `65535` for that timestamp, mostly clouds and sensor gaps.

Pixels are binned to a coarse grid for plotting; each cell shows whether valid or invalid pixels are the majority.

In [ ]:
BIN_SIZE_M = 200.0
valid_mask_global = (xs_global != XYS_SENTINEL) & (ys_global != XYS_SENTINEL)

n_bin_cols = int(np.ceil((TILE_MAX_X - TILE_MIN_X) / BIN_SIZE_M))
n_bin_rows = int(np.ceil((TILE_MAX_Y - TILE_MIN_Y) / BIN_SIZE_M))

bin_col = np.zeros(len(xs_global), dtype=np.int32)
bin_row = np.zeros(len(ys_global), dtype=np.int32)
bin_col[valid_mask_global] = ((xs_global[valid_mask_global] - TILE_MIN_X) // BIN_SIZE_M).astype(np.int32)
bin_row[valid_mask_global] = ((TILE_MAX_Y - ys_global[valid_mask_global]) // BIN_SIZE_M).astype(np.int32)
np.clip(bin_col, 0, n_bin_cols - 1, out=bin_col)
np.clip(bin_row, 0, n_bin_rows - 1, out=bin_row)

flat_bin_idx_valid = (bin_row[valid_mask_global] * n_bin_cols + bin_col[valid_mask_global]).astype(np.int64)
n_bins = n_bin_rows * n_bin_cols

total_per_bin = np.bincount(flat_bin_idx_valid, minlength=n_bins).reshape(n_bin_rows, n_bin_cols)
footprint = total_per_bin > 0
print(f"Bin grid: {n_bin_rows} rows x {n_bin_cols} cols  |  bin size {BIN_SIZE_M:.0f} m")
print(f"Footprint bins with data: {footprint.sum():,} of {n_bins:,}")

In [ ]:
def status_grid_for_timestamp(t_idx):
    with h5py.File(HDF5_PATH, "r") as f:
        vals = f["values"][t_idx, BAND_FOR_VALIDITY, :]
    valid_pixel = ((vals != NODATA_VAL) & valid_mask_global)[valid_mask_global].astype(np.int32)
    valid_per_bin = np.bincount(flat_bin_idx_valid, weights=valid_pixel,
                                minlength=n_bins).reshape(n_bin_rows, n_bin_cols)
    invalid_per_bin = total_per_bin - valid_per_bin
    grid = np.full((n_bin_rows, n_bin_cols), -1, dtype=np.int8)
    has_data = total_per_bin > 0
    grid[has_data & (valid_per_bin >= invalid_per_bin)] = 1
    grid[has_data & (valid_per_bin <  invalid_per_bin)] = 0
    return grid

def render_panel(ax, t_idx, grid):
    cmap = ListedColormap(["#ffffff", "#d4d4d4", "#2ca02c"])
    norm = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)
    extent = (TILE_MIN_X, TILE_MIN_X + n_bin_cols * BIN_SIZE_M,
              TILE_MAX_Y - n_bin_rows * BIN_SIZE_M, TILE_MAX_Y)
    ax.imshow(grid, cmap=cmap, norm=norm, extent=extent, origin="upper", interpolation="nearest")
    ax.contour(footprint.astype(float), levels=[0.5], extent=extent, origin="upper",
               colors="black", linewidths=0.6)
    cat_series = df_complete.loc[df_complete["date"] == timestamps[t_idx], "category"]
    cat = cat_series.iloc[0] if len(cat_series) else ""
    ax.set_title(f"{timestamps[t_idx].date()}  cloud {cloud_cover_pct[t_idx]}%\n{cat}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

START_T_IDX = 0
N_PANELS    = 20
sel = np.argsort(timestamps)[START_T_IDX : START_T_IDX + N_PANELS]

fig, axes = plt.subplots(4, 5, figsize=(18, 12))
for ax, t_idx in zip(axes.ravel(), sel):
    grid = status_grid_for_timestamp(int(t_idx))
    render_panel(ax, int(t_idx), grid)

legend_elems = [
    Patch(facecolor="#2ca02c", edgecolor="k", label="valid majority"),
    Patch(facecolor="#d4d4d4", edgecolor="k", label="invalid majority"),
    Patch(facecolor="#ffffff", edgecolor="k", label="outside footprint"),
]
fig.legend(handles=legend_elems, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.01), frameon=False)
fig.suptitle(f"T29TPG, 20 consecutive timestamps from t={START_T_IDX}", y=0.995, fontsize=13)
plt.tight_layout(); plt.show()

---
## 10. Vegetation indices time series for a sample chunk

Picks the densest chunk and traces NDVI, NDWI, and NBR across the year. Solid markers are usable scenes (`full_clear` or `partial_clear`); faded crosses are edge passes or cloudy scenes.

Indices:

* NDVI = (B8 − B4) / (B8 + B4)
* NDWI = (B8 − B11) / (B8 + B11)
* NBR  = (B8 − B12) / (B8 + B12)

In [ ]:
ci  = int(np.argmax(chip_pixel_count_arr))
lo, hi = ci * chunk_pixel_len, (ci + 1) * chunk_pixel_len
sx, sy = xs_global[lo:hi], ys_global[lo:hi]
mloc = (sx != XYS_SENTINEL) & (sy != XYS_SENTINEL)
print(f"Chunk {ci}  chip_x_bin={chip_x_bin[ci]} chip_y_bin={chip_y_bin[ci]}  valid pixels={int(mloc.sum()):,}")

b_idx = {b: band_names.index(b) for b in ["B4", "B8", "B11", "B12"]}

with h5py.File(HDF5_PATH, "r") as f:
    v = f["values"][:, list(b_idx.values()), lo:hi].astype(np.float32)   # (T, 4, chunk)
v[v == NODATA_VAL] = np.nan
v[:, :, ~mloc] = np.nan
B4, B8, B11, B12 = v[:,0,:], v[:,1,:], v[:,2,:], v[:,3,:]
ndvi = (B8 - B4)  / (B8 + B4)
ndwi = (B8 - B11) / (B8 + B11)
nbr  = (B8 - B12) / (B8 + B12)

df_idx = pd.DataFrame({
    "date": timestamps,
    "NDVI": np.nanmedian(ndvi, axis=1),
    "NDWI": np.nanmedian(ndwi, axis=1),
    "NBR":  np.nanmedian(nbr,  axis=1),
    "category": df_complete.set_index("date").loc[timestamps, "category"].values,
}).sort_values("date").reset_index(drop=True)

good = df_idx["category"].isin(["full_clear", "partial_clear"])
fig, ax = plt.subplots(figsize=(12, 4))
for col, color in [("NDVI","seagreen"), ("NDWI","steelblue"), ("NBR","darkorange")]:
    ax.plot(df_idx.loc[good, "date"], df_idx.loc[good, col], "o-", label=col, color=color, alpha=0.9)
    ax.plot(df_idx.loc[~good, "date"], df_idx.loc[~good, col], "x", color=color, alpha=0.3)
ax.axhline(0, color="gray", lw=0.5); ax.set_ylim(-0.5, 1.0)
ax.set_ylabel("Chunk median index value")
ax.set_title(f"Vegetation indices, chunk {ci} (chip_x={chip_x_bin[ci]}, chip_y={chip_y_bin[ci]})")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()